# Chapter 04-02 · Baselines first, always

**Label:** Core  |  **Time:** ~45 minutes  |  **Difficulty:** easy to do, uncomfortable to read

**Prerequisites:** 04-01 for framing, 03-01 for the mean and the median, 03-08 for loss versus metric.

**Position in the learning path:** module 04, chapter 2 of 8.

---

## Why this matters

04-01 ended on a number that should have been uncomfortable. A logistic regression that had genuinely
learned something scored **84.71%** accuracy. Predicting that nobody ever leaves scored **86.62%**.

That was not bad luck and it is not rare. It is the ordinary condition of an imbalanced problem, and the
only defence is to know what "doing nothing" scores **before** you fit anything.

This chapter builds those do-nothing predictors properly and then does something that most courses avoid:
it lets them win. A random forest with 200 trees will lose to a one-line baseline on this data, on both
metrics, and no amount of extra capacity will rescue it. That is a real outcome, it happens constantly in
practice, and **the only reason anyone ever finds out is that they computed a baseline.**

## What you will be able to do

- Build the four standard baselines for a regression and the three for a classification
- Compute a **skill score** and read a negative one without flinching
- Explain why a "predict last month's value" baseline can be worse than a constant
- Say what a metric means only in comparison to something, and to what
- Price a model in accuracy *and* in cost, and argue for the boring option
- Report a result in a form that cannot mislead

## Warm-up: retrieve, do not reread

1. What are the five framing questions?
2. Which constant minimises total absolute error, and which minimises total squared error?
3. In 04-01, why was the honest model's accuracy worse than the constant's?

<br>

*Answers: (1) unit, target, prediction time, horizon, availability. (2) the median and the mean
respectively - 03-01. (3) at a 13.44% base rate, always saying "no" is right 86.56% of the time, and the
model traded some of those correct "no"s for correct "yes"es.*

## The task

Same gym panel as 04-01. A new question, framed the same way: **for a member who is with us this month,
how many times will they visit next month?**

- **Unit:** one member-month
- **Target:** `next_visits`, that member's visits in the following month
- **Prediction time:** the end of the current month
- **Horizon:** one month
- **Availability:** anything from this month or earlier
- **Split:** months up to 16 to train, months 17 onward to test - chronological, because the model will
  be used on future months and 04-04 explains why a random split would flatter it

The decision it feeds: staffing. The gym needs to know expected attendance to roster instructors, so
being wrong by two visits a month per member matters and being wrong by a tenth of a visit does not.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# SYNTHETIC. The gym panel from 04-01.
def load_members():
    rng = np.random.default_rng(41)
    n = 600
    join_month = rng.integers(1, 13, n)
    commitment = rng.beta(2.0, 2.0, n)
    rows = []
    for member in range(n):
        drifting = 0.0
        base_visits = 2 + 10 * commitment[member]
        for month in range(join_month[member], 25):
            drifting += rng.normal(0.0, 0.35)
            visits = max(0, int(round(rng.poisson(max(0.2, base_visits - drifting)))))
            tickets = int(rng.random() < 0.05 + 0.10 * (visits == 0))
            hazard = 1 / (1 + np.exp(3.0 + 2.5 * commitment[member] - 0.55 * max(0, 4 - visits)))
            cancelled = int(rng.random() < hazard)
            rows.append((member + 1, month, visits, tickets, cancelled))
            if cancelled:
                break
    return pd.DataFrame(rows, columns=["member_id", "month", "visits", "tickets", "cancelled"])


panel = load_members().sort_values(["member_id", "month"])

# the target and the two features every baseline needs, all built from the past
panel["next_visits"] = panel.groupby("member_id").visits.shift(-1)
table = panel.dropna(subset=["next_visits"]).copy()
table["previous_visits"] = table.groupby("member_id").visits.shift(1)
table["mean_so_far"] = (table.groupby("member_id").visits
                        .expanding().mean().reset_index(level=0, drop=True))
table = table.dropna(subset=["previous_visits"])

table["months_of_history"] = table.groupby("member_id").cumcount() + 1

train = table[table.month <= 16]
test = table[table.month > 16]
actual = test.next_visits.to_numpy()

print("rows usable: %d   train (months <= 16) %d   test (months 17+) %d"
      % (len(table), len(train), len(test)))
print("next_visits: mean %.2f, median %.2f, sd %.2f"
      % (train.next_visits.mean(), train.next_visits.median(), train.next_visits.std()))

## The four baselines

A **baseline** is a prediction rule so simple that nobody could call it a model. Its job is not to be
good. Its job is to be **the thing your model has to beat in order to have been worth building.**

For a regression there are four worth computing every time, and they take one line each:

| Baseline | Rule | What beating it proves |
|---|---|---|
| **Global mean** | predict the same number for everyone | you have found *any* signal at all |
| **Global median** | predict the same number for everyone | the same, for an absolute-error metric |
| **Persistence** | predict what happened last time | you beat "nothing changes", the hardest baseline in most time series |
| **Per-entity mean** | predict this member's own average so far | you have learned something beyond *who* the member is |

The last one is the one people forget, and on this data it is the one that wins.

**Predict before running:** which of the four will be best? And how close will they be?

In [ ]:
def mean_absolute(prediction):
    return float(np.mean(np.abs(actual - prediction)))


def mean_squared(prediction):
    return float(np.mean((actual - prediction) ** 2))


baselines = {
    "global mean (%.2f)" % train.next_visits.mean(): np.full(len(test), train.next_visits.mean()),
    "global median (%.2f)" % train.next_visits.median(): np.full(len(test), train.next_visits.median()),
    "persistence: this month's visits": test.visits.to_numpy(),
    "this member's mean so far": test.mean_so_far.to_numpy(),
}

rows = [{"baseline": name, "MAE": round(mean_absolute(p), 4), "MSE": round(mean_squared(p), 4)}
        for name, p in baselines.items()]
print(pd.DataFrame(rows).to_string(index=False))

Two results here are worth more than the ranking.

**The per-entity mean wins by a distance.** 2.3640 against 2.9688 for the best constant - a 20%
reduction from one line of code that contains no model, no fitting and no parameters. It works because
members differ enormously from each other and rather less from themselves: knowing *which member* it is
tells you most of what there is to know.

**Persistence is worse than a constant.** 3.0482 against 2.9688. That is the surprising one, and it is
worth understanding because the reasoning generalises.

In [ ]:
compare = pd.DataFrame({
    "this month's visits": test.visits.to_numpy(),
    "member's mean so far": test.mean_so_far.to_numpy(),
    "next month's visits": actual,
})
print("how much each predictor jumps around, and how far it sits from the truth:")
print("  sd of 'this month's visits'    %.3f" % compare["this month's visits"].std())
print("  sd of 'member's mean so far'   %.3f" % compare["member's mean so far"].std())
print()
print("  correlation with next month: this month %.3f, mean so far %.3f"
      % (compare["this month's visits"].corr(compare["next month's visits"]),
         compare["member's mean so far"].corr(compare["next month's visits"])))
print()
print("  months of history behind each running mean, on average: %.1f"
      % test.months_of_history.mean())

**Both predictors are aiming at the same thing - the member's underlying visiting rate - and one of them
is a sample of size one.**

A member who genuinely averages 7 visits a month will record 5 one month and 9 the next, because
attendance is a count that fluctuates. Persistence takes that single noisy number as its estimate. The
running mean averages a dozen of them. This is 03-02's square-root law arriving in a place nobody
expects it: **the baseline that uses more data to estimate the same quantity wins, and using the most
recent value is not the same as using the most relevant information.**

The general lesson: **persistence is strong when the thing you are predicting moves slowly and is
measured precisely** - tomorrow's temperature, next quarter's headcount - and weak when each observation
is a noisy draw around a stable level. Compute it either way. It costs a line, and which of the two
worlds you are in is not always obvious in advance.

## One member, four baselines

Before the models, look at what these rules actually do to a single member. It makes the ranking obvious
in a way the table cannot.

In [ ]:
example = test.member_id.value_counts().index[0]
one = table[table.member_id == example]

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(one.month, one.next_visits, "o-", color="#000000", linewidth=1.6, label="what actually happened")
ax.plot(one.month, one.visits, "s--", color="#D55E00", markersize=4, alpha=0.8,
        label="persistence: this month's visits")
ax.plot(one.month, one.mean_so_far, color="#0072B2", linewidth=2, label="this member's mean so far")
ax.axhline(train.next_visits.median(), color="#009E73", linestyle=":", linewidth=2,
           label="global median (%.1f)" % train.next_visits.median())
ax.axvline(16.5, color="#999999")
ax.text(16.7, ax.get_ylim()[0] + 0.2, "test starts", color="#666666", fontsize=9)
ax.set_xlabel("month")
ax.set_ylabel("visits next month")
ax.set_title("Member %d. The flat blue line is the baseline that beats a random forest" % example)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

The orange line is persistence, and you can see it chasing every bounce - it copies this month's noise
into next month's prediction. The blue line is the running mean, which absorbs the same bounces and keeps
pointing at the member's actual level. The green line is where you would be with no per-member
information at all.

**Everything a model could add has to be added on top of the blue line.** That is the bar.

## Now the models

Two of them, given every column the baselines used and two more besides. Ordinary linear regression, and
a random forest with 200 trees.

**Predict before running:** by how much will they beat 2.3640?

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

columns = ["visits", "previous_visits", "mean_so_far", "tickets", "month"]

linear = LinearRegression().fit(train[columns], train.next_visits)
forest = RandomForestRegressor(n_estimators=200, random_state=0, min_samples_leaf=5)
forest.fit(train[columns], train.next_visits)

predictions = dict(baselines)
predictions["linear regression, 5 columns"] = linear.predict(test[columns])
predictions["random forest, 200 trees"] = forest.predict(test[columns])

best_baseline = min(mean_absolute(p) for p in baselines.values())

scoreboard = pd.DataFrame([
    {"predictor": name,
     "MAE": round(mean_absolute(p), 4),
     "MSE": round(mean_squared(p), 4),
     "skill vs best baseline": "%+.2f%%" % (100 * (best_baseline - mean_absolute(p)) / best_baseline)}
    for name, p in predictions.items()])
print(scoreboard.to_string(index=False))

## Both models are worse than the one-line baseline

Linear regression: **-1.49%**. Random forest: **-3.02%**. Negative skill means the model is worse than
the thing it was supposed to improve on.

Three defences you might reach for, and what the data says about each.

**"It is the metric."** No. The linear model was fitted to minimise squared error, so squared error is
the measure most favourable to it - and it loses there too, by **1.95%**, with the forest losing by
**7.10%**. Worth checking, because 03-08's point about the loss and the metric being different things is
real and this would have been a legitimate explanation. It just is not this one.

**"It needs more capacity."** No.

In [ ]:
rows = []
for trees in [10, 50, 200, 800]:
    bigger = RandomForestRegressor(n_estimators=trees, random_state=0, min_samples_leaf=5)
    bigger.fit(train[columns], train.next_visits)
    rows.append({"trees": trees, "MAE": round(mean_absolute(bigger.predict(test[columns])), 4)})
capacity = pd.DataFrame(rows)
capacity["still behind the baseline by"] = ((capacity.MAE - best_baseline).round(4))
print(capacity.to_string(index=False))
print()
print("the baseline, for reference: %.4f" % best_baseline)

Eighty times the trees moves the MAE from 2.4782 to 2.4338 and never reaches 2.3640. **The gap is not a
capacity problem, so throwing capacity at it does not close it.**

**"Then the features are wrong."** Closer, and this is the useful reading. The models were handed
`mean_so_far` - the winning baseline itself, as a column - and still did worse. What they did with it was
dilute it: they fitted coefficients on four other columns using 4,830 training rows from months 1-16, and
those coefficients carry sampling noise that the baseline, having no parameters, cannot have. **A model
with parameters can overfit. A model with no parameters cannot.**

That is the whole mechanism, and it is why the comparison is not a formality:

> **Every parameter you fit is a chance to be wrong in a new way. A baseline is worth beating precisely
> because it has no chances.**

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.6))

names = list(predictions)
values = [mean_absolute(p) for p in predictions.values()]
colours = ["#999999"] * 4 + ["#0072B2", "#D55E00"]
left.barh(range(len(names)), values, color=colours)
left.axvline(best_baseline, color="#000000", linestyle="--", linewidth=1.5)
left.text(best_baseline + 0.02, -0.35, "best baseline", fontsize=9)
left.set_yticks(range(len(names)))
left.set_yticklabels([n.replace(": ", ":\n") for n in names], fontsize=8)
left.set_xlim(2.0, 3.2)
left.set_xlabel("mean absolute error (lower is better)")
left.set_title("Grey is free. Blue and orange are not")
left.invert_yaxis()

skills = [100 * (best_baseline - v) / best_baseline for v in values]
right.barh(range(len(names)), skills,
           color=["#009E73" if s > 0 else "#D55E00" for s in skills])
right.axvline(0, color="#000000", linewidth=1.2)
for index, value in enumerate(skills):
    inside = value < -5
    right.text(value + 0.7 if inside else value - 0.7, index, "%+.2f%%" % value,
               va="center", ha="left" if inside else "right", fontsize=8,
               color="#ffffff" if inside else "#000000")
right.set_yticks(range(len(names)))
right.set_yticklabels([])
right.set_xlim(-32, 4)
right.set_xlabel("skill over the best baseline (%)")
right.set_title("Nothing is to the right of zero")
right.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3), sharey=True)
for ax, (label, prediction) in zip(axes, [
        ("global median", predictions["global median (%.2f)" % train.next_visits.median()]),
        ("per-member mean (the baseline)", predictions["this member's mean so far"]),
        ("random forest, 200 trees", predictions["random forest, 200 trees"])]):
    ax.scatter(prediction, actual, s=6, alpha=0.12, color="#0072B2")
    limits = [-1, 22]
    ax.plot(limits, limits, color="#000000", linewidth=1, linestyle=":")
    ax.set_xlim(0, 16)
    ax.set_ylim(-1, 22)
    ax.set_xlabel("predicted visits")
    ax.set_title("%s\nMAE %.4f" % (label, mean_absolute(prediction)), fontsize=10)
axes[0].set_ylabel("what actually happened")
fig.suptitle("A flat predictor, a good one, and a model that spreads out without improving",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

The left panel is a constant: one vertical stripe, because every prediction is the same number. The middle
panel spreads along the diagonal - the baseline really does distinguish members. The right panel spreads
*further* and is not closer to the line, which is the picture of a model that has learned to vary its
predictions without learning to be right.

## The skill score

The right-hand panel is the number worth reporting, and it has a name:

> **skill = (baseline error - model error) / baseline error**

It answers "how much of the baseline's error did the model remove?" - 0% means it matched the baseline,
100% means it is perfect, and negative means it made things worse.

**A metric alone is uninterpretable and a skill score is not.** "MAE 2.4353" could be excellent or
useless; you cannot tell without knowing what the target's spread is or what free rules achieve.
"-3.02% against a per-member mean" is a complete statement. Two habits follow:

- **Always report the baseline you beat, by name.** "3% better than a random forest" is not a claim about
  anything; "12% better than a per-member mean" is.
- **Name the *strongest* baseline, not a convenient one.** Against the global median this forest shows
  **+17.97%** skill, which sounds like a result. The same model, same data, same day. Choosing which
  baseline to quote is the easiest way to mislead in this entire field, and it usually is not deliberate
  - people compute the easy baseline, beat it, and stop.

## The other half of the price: cost

In [ ]:
def seconds(build, repeats=3):
    started = time.perf_counter()
    for _ in range(repeats):
        build()
    return (time.perf_counter() - started) / repeats


baseline_cost = seconds(lambda: test.mean_so_far.to_numpy(), 20)
linear_cost = seconds(lambda: LinearRegression().fit(train[columns], train.next_visits)
                      .predict(test[columns]))
forest_cost = seconds(lambda: RandomForestRegressor(n_estimators=200, random_state=0,
                                                    min_samples_leaf=5)
                      .fit(train[columns], train.next_visits).predict(test[columns]), 2)

print("%-28s %10s  %12s  %s" % ("", "seconds", "MAE", "skill"))
for name, cost, error in [("per-member mean", baseline_cost, best_baseline),
                          ("linear regression", linear_cost, mean_absolute(predictions["linear regression, 5 columns"])),
                          ("random forest, 200 trees", forest_cost, mean_absolute(predictions["random forest, 200 trees"]))]:
    print("%-28s %10.5f  %12.4f  %+.2f%%"
          % (name, cost, error, 100 * (best_baseline - error) / best_baseline))
print()
print("the forest costs about %.0f times more compute than the baseline, to do worse"
      % (forest_cost / baseline_cost))

Accuracy is only one column of the invoice. The other columns:

- **compute**, which the cell above prices
- **latency** at prediction time, which decides whether a model can sit inside a request
- **dependencies** - the forest needs scikit-learn, a pinned version, and a serialised artefact that must
  be rebuilt when the library changes; the baseline needs `groupby`
- **explanation** - somebody will ask why a member was predicted 3 rather than 8, and one of these
  answers that in a sentence
- **failure modes** - the baseline degrades gracefully for a new member (no history, fall back to the
  global mean); the forest extrapolates confidently and wrongly outside its training range

**None of that appears in a metric.** A model that wins by 1% and costs all of the above is usually the
wrong choice, and the number that would have told you is the skill score, not the MAE.

This is not an argument against models. It is an argument for **knowing the size of what you are buying
before you decide what to pay for it.**

## Baselines for a classification

The same discipline, three rules instead of four. Back to 04-01's churn task - wall at month 12, horizon
six months, base rate 13.44%.

| Baseline | Rule | Why you need it |
|---|---|---|
| **Majority class** | predict the commonest label for everyone | the accuracy any model must exceed to have done anything |
| **Random, matching the base rate** | guess "yes" 13.44% of the time | the AUC floor: exactly 0.5 |
| **One rule** | threshold on the single most promising column | the score a thirty-second answer achieves |

The third is the one that saves projects. It is astonishing how often a single threshold on one column
gets most of the available signal - and knowing that *before* the pipeline is built changes what gets
built.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

CUT, HORIZON = 12, 6
active = panel[(panel.month == CUT) & (panel.cancelled == 0)].member_id.unique()
history = panel[panel.member_id.isin(active) & (panel.month <= CUT)]
future = panel[panel.member_id.isin(active) & (panel.month > CUT)]
left = future[(future.month <= CUT + HORIZON) & (future.cancelled == 1)].member_id.unique()
churn = pd.Series(np.isin(active, left).astype(int), index=active)

features = pd.DataFrame({
    "visits_at_cut": history[history.month == CUT].set_index("member_id").visits.reindex(active),
    "mean_visits_last_3": history[history.month > CUT - 3].groupby("member_id").visits.mean().reindex(active),
    "tenure_months": history.groupby("member_id").size().reindex(active),
    "tickets_so_far": history.groupby("member_id").tickets.sum().reindex(active),
})

train_X, test_X, train_y, test_y = train_test_split(
    features, churn, test_size=0.3, random_state=0, stratify=churn)
centre, spread = train_X.mean(), train_X.std()
logistic = LogisticRegression(max_iter=2000).fit((train_X - centre) / spread, train_y)
probability = logistic.predict_proba((test_X - centre) / spread)[:, 1]

print("%-34s %10s %8s" % ("", "accuracy", "AUC"))
print("%-34s %10.4f %8.4f" % ("majority class ('nobody leaves')", 1 - test_y.mean(), 0.5))
print("%-34s %10.4f %8.4f" % ("logistic regression, 4 columns",
                              accuracy_score(test_y, probability > 0.5),
                              roc_auc_score(test_y, probability)))
print()
print("one rule, on visits in month 12:")
for threshold in [0, 1, 2, 3, 4]:
    rule = (test_X.visits_at_cut <= threshold).astype(int)
    print("  flag if visits <= %d : accuracy %.4f  AUC %.4f  flags %2d of %d members"
          % (threshold, accuracy_score(test_y, rule), roc_auc_score(test_y, rule),
             int(rule.sum()), len(rule)))

In [ ]:
thresholds = [0, 1, 2, 3, 4]
rule_accuracy, rule_auc, flagged = [], [], []
for threshold in thresholds:
    rule = (test_X.visits_at_cut <= threshold).astype(int)
    rule_accuracy.append(accuracy_score(test_y, rule))
    rule_auc.append(roc_auc_score(test_y, rule))
    flagged.append(int(rule.sum()))

fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.3))

left.plot(thresholds, rule_accuracy, "o-", color="#D55E00", label="one-rule accuracy")
left.axhline(1 - test_y.mean(), color="#000000", linestyle="--",
             label="majority class (%.4f)" % (1 - test_y.mean()))
left.axhline(accuracy_score(test_y, probability > 0.5), color="#0072B2", linestyle=":",
             label="logistic regression")
left.set_xlabel("flag if visits at month 12 <= threshold")
left.set_ylabel("accuracy")
left.set_title("Accuracy: nothing beats saying 'nobody leaves'", fontsize=11)
left.legend(fontsize=8)

right.plot(thresholds, rule_auc, "o-", color="#D55E00", label="one-rule AUC")
right.axhline(0.5, color="#000000", linestyle="--", label="chance")
right.axhline(roc_auc_score(test_y, probability), color="#0072B2", linestyle=":",
              label="logistic regression (%.4f)" % roc_auc_score(test_y, probability))
for axis in (left, right):
    axis.set_xticks(thresholds)
    axis.set_xticklabels(["%d\n(%d flagged)" % (t, c) for t, c in zip(thresholds, flagged)],
                         fontsize=8)
right.set_xlabel("flag if visits at month 12 <= threshold")
right.set_ylabel("AUC")
right.set_title("AUC: the rule and the model are neck and neck", fontsize=11)
right.legend(fontsize=8, loc="center right")

plt.tight_layout()
plt.show()

Read that carefully, because there are three separate lessons in it.

**1. The majority class wins on accuracy, again.** 0.8662 against the model's 0.8471. Every one-rule
threshold also loses to it. **On this problem, accuracy is a metric no useful model can win**, and the
only reason to compute it is to prove that.

**2. The AUC tells the opposite story, and it is the true one.** The model scores 0.6359 against a floor
of 0.5. The one-rule thresholds get to 0.6425 at `visits <= 4` - genuinely competitive with the model,
from a rule you could explain to the front desk. That comparison is the value of the exercise: **the
model is worth roughly one threshold on one column**, which is a fact worth knowing before anyone builds
a pipeline around it.

**3. A threshold that flags almost nobody looks fine on accuracy and is useless.** `visits <= 0` flags
**one member out of 157** and scores 0.8599 accuracy - barely below the majority class, because it is
almost the majority class. Its AUC of 0.4963 says it is worthless. **Accuracy rewards timidity on an
imbalanced problem**, and a rule that abstains scores well by declining to be wrong.

Together those give the rule that matters:

> **Pick the metric with the baseline, not after the model.** If the metric you chose cannot separate a
> useful model from a constant, it is the wrong metric, and finding that out costs one cell.

Module 06 and 07 build the proper machinery for this. What belongs here, in the workflow, is the habit:
**compute the baselines and the metric together, first, and require that a good model could in principle
win.**

## How to report a result

Bad, and extremely common:

> *"The model achieves 84.7% accuracy and an MAE of 2.44."*

Neither number means anything. There is no baseline, no denominator, no statement of what was achievable.

Better:

> *"On months 17-23, held out chronologically: predicting each member's own running mean gives MAE
> 2.3640. A random forest on five columns gives 2.4353 - **3.02% worse**. Against the global median
> (2.9688), the same forest is 17.97% better, which is the comparison I would have reported had I not
> computed the stronger baseline. Recommendation: ship the running mean."*

Everything checkable is stated: the split, the baseline by name, the direction, and the fact that a more
flattering comparison existed and was rejected. **A result that does not name its baseline is not a
result.**

## Common misconceptions

**"Baselines are a formality before the real work."**
Both models here lost to one. That outcome is only visible to somebody who computed the baseline, and it
is the correct recommendation.

**"A baseline is the simplest model."**
It is the strongest rule that involves no fitting. Per-member means, last-value, and a single threshold
are all baselines, and the per-entity one is usually the hardest to beat.

**"If my model beats the baseline, I am done."**
By how much, and at what cost? A 1% improvement that adds a dependency, a serialised artefact and a
retraining schedule is a bad trade, and skill without cost is half an answer.

**"A high accuracy means a good model."**
0.8662 of it was available for free here. The question is never the level, it is the distance above the
floor.

**"Persistence is always a strong baseline."**
It is strong when observations are precise and the quantity moves slowly. Here each observation is a
noisy count, so last month's value lost to a constant.

**"Negative skill means I made a mistake."**
It usually means the baseline is good. That is information, not an error - and it is the cheapest
information you will get all project.

**"I will compute a baseline if the model looks disappointing."**
Then the baseline is being used to excuse a result rather than to judge it, and you will never discover
the cases where a *good-looking* model was still worse than doing nothing. Which is this chapter.

In [ ]:
ladder = sorted([(name, mean_absolute(prediction)) for name, prediction in predictions.items()],
                key=lambda row: -row[1])

fig, ax = plt.subplots(figsize=(10, 4.8))
for step, (name, error) in enumerate(ladder):
    is_baseline = "regression" not in name and "forest" not in name
    ax.add_patch(plt.Rectangle((step, 0), 0.94, error,
                               facecolor="#999999" if is_baseline else "#D55E00",
                               edgecolor="white", linewidth=2))
    ax.text(step + 0.47, error + 0.04, "%.4f" % error, ha="center", fontsize=9)
    ax.text(step + 0.47, 0.12, name.replace(": ", ":\n").replace(" (", "\n("),
            ha="center", va="bottom", fontsize=7.5, color="white", rotation=90)
ax.axhline(best_baseline, color="#000000", linestyle="--", linewidth=1.6)
ax.text(0.05, best_baseline + 0.07, "the bar set by a free rule: %.4f" % best_baseline,
        ha="left", fontsize=9.5, fontweight="bold")
ax.set_xlim(-0.15, len(ladder))
ax.set_ylim(0, 3.35)
ax.set_xticks([])
ax.set_ylabel("mean absolute error (lower is better)")
ax.set_title("Grey bars cost nothing. The two orange ones cost a project, and neither clears the bar",
             fontsize=11.5)
plt.tight_layout()
plt.show()

That is the chapter in one frame: **six predictors, four of them free, and the bar is set by a free one.**

## Exercises

Solutions: `solutions/04_workflow/04-02_baselines_solutions.ipynb`.

### Quick understanding

**E1.** Name the four regression baselines and the three classification baselines from this chapter, and
say in one sentence what beating each one proves.

**E2.** Define the skill score, and say what 0%, 100% and -5% each mean.

**E3.** Why is a model with parameters able to lose to a rule with none, even when the rule's output was
given to it as a column?

### Hand calculation

**E4.** A baseline scores MAE 40.0 and a model scores MAE 34.0. Compute the skill. Then the model is
improved to MAE 30.0 - compute the new skill, and the percentage improvement over the *previous model*.
Say which of the two numbers you would put in a slide and why the other one is not wrong.

**E5.** A classification problem has 1,000 rows, 40 of them positive. Compute (a) the majority-class
accuracy, (b) the accuracy of a rule that flags 20 rows and gets 12 of them right, and (c) that rule's
precision and recall. Then say which of (a) and (b) is the more useful predictor and why the accuracies
disagree with your answer.

**E6.** Two teams report on the same dataset. Team A: "MAE 2.44, an 18% improvement." Team B: "MAE 2.36,
no improvement." Using this chapter's numbers, explain how both statements can be true and which team you
would rather have.

**E7.** A member has visited 6, 8, 5, 9, 3 times over five months. Compute what persistence and the
running mean each predict for month 6. The member then visits 4 times. Compute each rule's absolute
error, and say what a single month tells you about which rule is better.

### Coding

**E8.** Write `skill(baseline_error, model_error)` and a `report(name, prediction)` helper that prints
the metric and the skill against a named baseline in one line. Use it to reproduce this chapter's
scoreboard.

**E9.** Add a fifth baseline: **this member's mean over the last three months only**. Does a shorter
window beat the full running mean? Report the MAE and explain the result in terms of the trade-off
between using more data and using more relevant data.

**E10.** The forest lost. Give it a fair chance: drop `month` and `tickets`, keep only `mean_so_far` and
`visits`, and refit. Report the MAE. Does removing columns help, and what does your answer say about the
"more features is better" instinct?

**E11.** Build the **per-entity median** baseline instead of the mean, and compare on MAE. Predict which
should win before running it, using 03-01.

**E12.** For the classification task, write a loop that finds the single best one-rule threshold on each
of the four columns by AUC on the training set, then reports that rule's AUC on the test set. Compare
with the logistic regression. Which column wins, and by how much does the model beat the best rule?

### Interpretation

**E13.** A colleague reports that their fraud model achieves 99.2% accuracy. Fraud is 0.5% of
transactions. State what you know immediately, and the two numbers you would ask for.

**E14.** A demand-forecasting model beats a persistence baseline by 30% on MAE, but persistence beats it
on the ten largest-volume products, which are 70% of revenue. Say what you would recommend and what
additional number you would want.

### Debugging

**E15.** Your baseline scores *better* on the test set than on the training set. Give two explanations,
one benign and one that indicates a real problem.

**E16.** A colleague's "per-member mean" baseline scores MAE 0.31, far better than anything in this
chapter. Name the most likely cause and the one line you would look at.

### Exam and interview reasoning

**E17.** "What is the first thing you do on a new modelling problem?" Answer in under a minute, and be
ready for the follow-up: "what if the baseline wins?"

### Transfer to a different situation

**E18.** You are asked to forecast daily electricity demand for a city. Write down four baselines you
would compute before fitting anything, at least one of which uses the structure of the problem (hint:
demand at 6pm on a Tuesday). Say which you expect to be hardest to beat and why.

### Explain it to someone non-technical

**E19.** Your manager asks why you spent the first day of a modelling project "not building a model".
Answer in under 90 words, using this chapter's result.

### Optional challenge

**E20.** Build a **learning curve against the baseline**: fit the forest on 10%, 25%, 50% and 100% of the
training rows and plot its MAE with the baseline as a horizontal line. Does the curve suggest that more
data would eventually let the model win? State what the shape of the curve would need to look like for
"collect more data" to be the right recommendation - and whether it does.

In [ ]:
# Your workspace. In memory: panel, table, train, test, actual, baselines, predictions,
# best_baseline, columns, mean_absolute, mean_squared, features, churn, logistic.

## Mastery check

- [ ] Name and build four regression baselines and three classification baselines without notes
- [ ] Compute a skill score and state what its sign means
- [ ] Explain why a parameter-free rule can beat a fitted model
- [ ] Say when persistence is strong and when it is weak, with the mechanism
- [ ] Spot a metric that no useful model could win on a given problem
- [ ] Report a result in a form that names its baseline and its split
- [ ] Argue for shipping a baseline, out loud, without embarrassment

## What should now feel instinctive

- Computing baselines in the first hour, before any model exists
- Reaching for the per-entity baseline, not just the constant
- Asking "compared to what?" of every metric anybody quotes, including your own
- Quoting the strongest baseline you computed, not the one that flatters the model
- Reading a negative skill score as a finding rather than a failure

## Flashcards

| Front | Back |
|---|---|
| Baseline | A prediction rule with no fitted parameters. The bar a model must clear to be worth building |
| Four regression baselines | Global mean, global median, persistence, per-entity mean |
| Three classification baselines | Majority class, base rate (AUC 0.5), best single-column rule |
| Skill score | `(baseline error - model error) / baseline error`. Negative means worse than free |
| The winner here | Per-member running mean, MAE 2.3640 |
| The forest here | MAE 2.4353, skill -3.02%, and -7.10% on squared error too |
| Why persistence lost | Each observation is a noisy count; one draw is a worse estimate than twelve |
| Why the models lost | Fitted parameters carry sampling noise; a rule with none cannot overfit |
| Same forest, weaker baseline | +17.97% against the global median. Choose the baseline honestly |
| Majority-class accuracy here | 0.8662, above anything a useful model scored |
| A rule that flags almost nobody | Scores well on accuracy, AUC 0.4963. Accuracy rewards timidity |
| How to report | Metric, split, and the named baseline. Without the baseline it is not a result |

## Next

**04-03 · Splitting I: train, validation and test.** This chapter kept saying "held out" and "on months
17-23" without justifying either. The comparison it rests on - baseline 2.3640 against forest 2.4353 -
is only meaningful because both were measured on rows neither had seen, and the module 03 assessment
already showed a model beating the noise it was fitted on.

The next chapter makes that precise: why a held-out set is the only evidence, why three splits are needed
rather than two once you start choosing between models, and what stratification protects when the
interesting class is rare. Then 04-04 breaks it, using exactly the member-month table this chapter built.